In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readCsvFile").getOrCreate()
Salesdf = spark.read.csv("Sales.csv",header=True,inferSchema=True)
Salesdf.show()
Salesdf.count()

+---------+-----------+----------+-----------+
|OrderDate|CustomerKey|ProductKey|SalesAmount|
+---------+-----------+----------+-----------+
| 1/1/2016|      11006|       490|   699.0982|
| 1/1/2016|      11007|       336|      34.99|
| 1/1/2016|      11003|       463|       32.6|
| 1/1/2016|      11005|       536|      29.99|
| 1/1/2016|      11003|       390|      53.99|
| 1/1/2016|      11004|       225|       8.99|
| 1/1/2016|      11007|       463|    1120.49|
| 1/1/2016|      11006|       336|       8.99|
| 1/1/2016|      11002|       463|      24.49|
| 1/1/2016|      11004|       225|       4.99|
| 1/1/2016|      11006|       536|       32.6|
| 1/1/2016|      11004|       490|      34.99|
| 1/1/2016|      11000|       540|       4.99|
| 1/1/2016|      11003|       490|       8.99|
| 1/1/2016|      11000|       490|    2443.35|
| 1/1/2016|      11001|       490|       2.29|
| 1/1/2016|      11003|       536|       4.99|
| 1/1/2016|      11002|       336|      53.99|
| 1/1/2016|  

51

In [27]:

bytes_read = Salesdf.rdd.map(lambda row: len(row[0].encode("utf-8"))).sum()
print(bytes_read)


408


In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readCsvFile").getOrCreate()
Productsdf = spark.read.csv("ProductsNew.csv",header=True,inferSchema=True)
Productsdf.show()
Productsdf.count()

+----------+--------------------+
|ProductKey|  EnglishProductName|
+----------+--------------------+
|       222|Sport-100 Helmet,...|
|       225|        AWC Logo Cap|
|       336|  Road-650 Black, 62|
|       390|Road-550-W Yellow...|
|       463|Half-Finger Glove...|
|       477|Water Bottle - 30...|
|       479|    Road Bottle Cage|
|       490|Short-Sleeve Clas...|
|       536|    ML Mountain Tire|
|       540|        HL Road Tire|
|       541|        HL Road Tire|
+----------+--------------------+



11

In [7]:
joindf = Salesdf.join(Productsdf,Salesdf.ProductKey==Productsdf.ProductKey)
joindf.show()
joindf.count()

+---------+-----------+----------+-----------+----------+--------------------+
|OrderDate|CustomerKey|ProductKey|SalesAmount|ProductKey|  EnglishProductName|
+---------+-----------+----------+-----------+----------+--------------------+
| 1/1/2016|      11006|       490|   699.0982|       490|Short-Sleeve Clas...|
| 1/1/2016|      11007|       336|      34.99|       336|  Road-650 Black, 62|
| 1/1/2016|      11003|       463|       32.6|       463|Half-Finger Glove...|
| 1/1/2016|      11005|       536|      29.99|       536|    ML Mountain Tire|
| 1/1/2016|      11003|       390|      53.99|       390|Road-550-W Yellow...|
| 1/1/2016|      11004|       225|       8.99|       225|        AWC Logo Cap|
| 1/1/2016|      11007|       463|    1120.49|       463|Half-Finger Glove...|
| 1/1/2016|      11006|       336|       8.99|       336|  Road-650 Black, 62|
| 1/1/2016|      11002|       463|      24.49|       463|Half-Finger Glove...|
| 1/1/2016|      11004|       225|       4.99|      

50

In [8]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")


'10485760b'

In [9]:
10485760/1024/1024

10.0

In [10]:
joindf.explain(True)

== Parsed Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Analyzed Logical Plan ==
OrderDate: string, CustomerKey: int, ProductKey: int, SalesAmount: double, ProductKey: int, EnglishProductName: string
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Optimized Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Filter isnotnull(ProductKey#257)
:  +- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Filter isnotnull(ProductKey#311)
   +- Relation [ProductKey#311,EnglishProductName#312] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [ProductKey#257], [ProductKey#311], Inner, BuildRight, false
   :- Filter isnotnull(ProductKey#25

In [12]:
spark.conf.get("spark.sql.broadcastTimeout")


'300000ms'

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast


In [16]:
joindf = Salesdf.join(broadcast(Productsdf),Salesdf.ProductKey==Productsdf.ProductKey)
joindf.show()
joindf.count()

+---------+-----------+----------+-----------+----------+--------------------+
|OrderDate|CustomerKey|ProductKey|SalesAmount|ProductKey|  EnglishProductName|
+---------+-----------+----------+-----------+----------+--------------------+
| 1/1/2016|      11006|       490|   699.0982|       490|Short-Sleeve Clas...|
| 1/1/2016|      11007|       336|      34.99|       336|  Road-650 Black, 62|
| 1/1/2016|      11003|       463|       32.6|       463|Half-Finger Glove...|
| 1/1/2016|      11005|       536|      29.99|       536|    ML Mountain Tire|
| 1/1/2016|      11003|       390|      53.99|       390|Road-550-W Yellow...|
| 1/1/2016|      11004|       225|       8.99|       225|        AWC Logo Cap|
| 1/1/2016|      11007|       463|    1120.49|       463|Half-Finger Glove...|
| 1/1/2016|      11006|       336|       8.99|       336|  Road-650 Black, 62|
| 1/1/2016|      11002|       463|      24.49|       463|Half-Finger Glove...|
| 1/1/2016|      11004|       225|       4.99|      

50

In [22]:
spark.conf.set("Spark.sql.autoBroadcastJoinThreshold",20*1024*1024)

In [23]:
spark.conf.get("Spark.sql.autoBroadcastJoinThreshold")

'20971520'

In [24]:
20971520/1024/1024

20.0

In [25]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1) # disabled it

In [26]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'-1'

In [29]:
spark.conf.set("Spark.sql.autoBroadcastJoinThreshold",30)

In [31]:
joindf = Salesdf.join((Productsdf),Salesdf.ProductKey==Productsdf.ProductKey)
joindf.show()
joindf.count()

+---------+-----------+----------+-----------+----------+--------------------+
|OrderDate|CustomerKey|ProductKey|SalesAmount|ProductKey|  EnglishProductName|
+---------+-----------+----------+-----------+----------+--------------------+
| 1/1/2016|      11002|       222|       2.29|       222|Sport-100 Helmet,...|
| 1/1/2016|      11007|       222|      49.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11000|       222|       2.29|       222|Sport-100 Helmet,...|
| 2/1/2016|      11003|       222|     539.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11000|       222|       4.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11007|       222|    2294.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11004|       222|       8.99|       222|Sport-100 Helmet,...|
| 1/1/2016|      11004|       225|       8.99|       225|        AWC Logo Cap|
| 1/1/2016|      11004|       225|       4.99|       225|        AWC Logo Cap|
| 1/1/2016|      11001|       225|       63.5|      

50

In [32]:
joindf.explain(True) # to check spark job 

== Parsed Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Analyzed Logical Plan ==
OrderDate: string, CustomerKey: int, ProductKey: int, SalesAmount: double, ProductKey: int, EnglishProductName: string
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Optimized Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Filter isnotnull(ProductKey#257)
:  +- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Filter isnotnull(ProductKey#311)
   +- Relation [ProductKey#311,EnglishProductName#312] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [ProductKey#257], [ProductKey#311], Inner
   :- Sort [ProductKey#257 ASC NULLS FIRST], false, 0
   : 

# 72 suffle hash algorithm

In [35]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50)
 #may i tried to take false instead if 50 but is show error 

In [36]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'50'

In [38]:
joindf2= Salesdf.join((Productsdf),Salesdf.ProductKey==Productsdf.ProductKey)
joindf2.show()
joindf2.count()

+---------+-----------+----------+-----------+----------+--------------------+
|OrderDate|CustomerKey|ProductKey|SalesAmount|ProductKey|  EnglishProductName|
+---------+-----------+----------+-----------+----------+--------------------+
| 1/1/2016|      11002|       222|       2.29|       222|Sport-100 Helmet,...|
| 1/1/2016|      11007|       222|      49.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11000|       222|       2.29|       222|Sport-100 Helmet,...|
| 2/1/2016|      11003|       222|     539.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11000|       222|       4.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11007|       222|    2294.99|       222|Sport-100 Helmet,...|
| 2/1/2016|      11004|       222|       8.99|       222|Sport-100 Helmet,...|
| 1/1/2016|      11004|       225|       8.99|       225|        AWC Logo Cap|
| 1/1/2016|      11004|       225|       4.99|       225|        AWC Logo Cap|
| 1/1/2016|      11001|       225|       63.5|      

50

In [39]:
joindf2.explain(True)

== Parsed Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Analyzed Logical Plan ==
OrderDate: string, CustomerKey: int, ProductKey: int, SalesAmount: double, ProductKey: int, EnglishProductName: string
Join Inner, (ProductKey#257 = ProductKey#311)
:- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Relation [ProductKey#311,EnglishProductName#312] csv

== Optimized Logical Plan ==
Join Inner, (ProductKey#257 = ProductKey#311)
:- Filter isnotnull(ProductKey#257)
:  +- Relation [OrderDate#255,CustomerKey#256,ProductKey#257,SalesAmount#258] csv
+- Filter isnotnull(ProductKey#311)
   +- Relation [ProductKey#311,EnglishProductName#312] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [ProductKey#257], [ProductKey#311], Inner
   :- Sort [ProductKey#257 ASC NULLS FIRST], false, 0
   : 

In [ ]:
#in contain of sort it present shuffle 
# it contain loweset size it built a hash tabke in memory and join to big dataset
